<a href="https://colab.research.google.com/github/FelipeFMMobile/finetunningaudio/blob/main/notebooks/01_qwen3_tts_clone_finetuning_loRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir LoRA no Colab"/></a>

# Pipeline Qwen3-TTS LoRA: adaptação leve para T4

Esta variante reaproveita a preparação e o zero-shot do notebook principal, mas substitui o fine-tuning completo por **LoRA (Low-Rank Adaptation)**. Os pesos originais ficam congelados e somente pequenas matrizes são treinadas, reduzindo gradientes e estados do otimizador para caber em uma T4 de 16 GB.

| Parte deste notebook | Referência do curso |
|---|---|
| Parte A — Preparação dos dados | `02_data_prep.ipynb` |
| Parte B — Clonagem zero-shot | `01_voice_cloning.ipynb` |
| Parte C — LoRA e teste | adaptação experimental de `03_finetune.ipynb` |

> LoRA para Qwen3-TTS ainda é uma extensão comunitária, não o caminho oficial básico. Preserve o notebook principal para comparar com full fine-tuning em L4/A100.


## 1. Preparação do Colab — variante LoRA para T4

Selecione **Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4**. Esta variante usa Qwen3-TTS 0.6B em FP32 porque FP16 é instável na arquitetura Turing da T4. O ganho de memória vem de congelar o modelo-base e treinar apenas os adaptadores LoRA.

Além do Qwen oficial, instalamos PEFT e uma implementação comunitária de LoRA fixada a revisões conhecidas. Isso torna o experimento reproduzível e evita mudanças silenciosas no código externo.


In [ ]:
import gc
import importlib
import importlib.util
import json
import os
import random
import shutil
import subprocess
import sys
import tarfile
from datetime import datetime, timezone
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA não encontrada. Selecione Ambiente de execução → "
        "Alterar tipo de ambiente de execução → GPU."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {GPU_NAME} | memória: {GPU_MEMORY_GB:.1f} GB")


In [ ]:
QWEN_REPO = Path("/content/Qwen3-TTS")
LORA_REPO = Path("/content/qwen3-tts-lora-finetuning")
QWEN_PIN = "0c6a7cbb6c8421a46332f8c2434c7825c4c855ef"
LORA_PIN = "4076c434e3bc51c928410f28f68a4f76f8f2e715"

if not QWEN_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/QwenLM/Qwen3-TTS.git", str(QWEN_REPO)], check=True)
    subprocess.run(["git", "-C", str(QWEN_REPO), "checkout", QWEN_PIN], check=True)
if not LORA_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/instavar/qwen3-tts-lora-finetuning.git", str(LORA_REPO)], check=True)
    subprocess.run(["git", "-C", str(LORA_REPO), "checkout", LORA_PIN], check=True)

QWEN_COMMIT = subprocess.check_output(["git", "-C", str(QWEN_REPO), "rev-parse", "HEAD"], text=True).strip()
if QWEN_COMMIT != QWEN_PIN:
    raise RuntimeError("/content/Qwen3-TTS contém outra revisão. Reinicie a sessão do Colab.")
if subprocess.check_output(["git", "-C", str(LORA_REPO), "rev-parse", "HEAD"], text=True).strip() != LORA_PIN:
    raise RuntimeError("/content/qwen3-tts-lora-finetuning contém outra revisão. Reinicie a sessão.")

lora_patch = LORA_REPO / "patches" / "0001-qwen3-tts-lora.patch"
lora_trainer = QWEN_REPO / "finetuning" / "sft_12hz_lora.py"
if not lora_trainer.exists():
    subprocess.run(["git", "-C", str(QWEN_REPO), "apply", "--check", str(lora_patch)], check=True)
    subprocess.run(["git", "-C", str(QWEN_REPO), "apply", str(lora_patch)], check=True)

infer_lora_script = QWEN_REPO / "finetuning" / "infer_lora_custom_voice.py"
infer_source = infer_lora_script.read_text(encoding="utf-8")
main_guard = 'if __name__ == "__main__":'
if infer_source.rstrip().endswith(main_guard):
    infer_source = infer_source.rstrip() + "\n    main()\n"
    infer_lora_script.write_text(infer_source, encoding="utf-8")
compile(infer_lora_script.read_text(encoding="utf-8"), str(infer_lora_script), "exec")

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "sox", "libsox-fmt-all"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(QWEN_REPO),
     "peft", "openai-whisper", "pydub", "soundfile", "accelerate", "tensorboard",
     "huggingface_hub", "safetensors"],
    check=True,
)
importlib.invalidate_caches()
if importlib.util.find_spec("qwen_tts") is None or importlib.util.find_spec("peft") is None:
    raise RuntimeError("Qwen3-TTS ou PEFT não ficou disponível. Reinicie a sessão e tente novamente.")
print("Qwen3-TTS:", QWEN_COMMIT)
print("Implementação LoRA:", LORA_PIN)


### 1.1 Configuração e entrada dos dados

Se você já baixou `*_dataset_preparado.tar.gz`, use `INPUT_MODE = "prepared"` e envie esse pacote nesta mesma T4. Assim, o notebook pula segmentação, Whisper e tokenização. Use `INPUT_MODE = "raw"` somente se estiver começando pela gravação WAV.

Os hiperparâmetros LoRA ficam concentrados nesta célula. Para o primeiro experimento, mantenha rank 8, alpha 16, escala 0,30 e batch 1.


In [ ]:
from google.colab import files

SPEAKER_NAME = "felipe"
INPUT_MODE = "raw"  # use "prepared" para importar o dataset nesta T4
RAW_AUDIO_FILENAME = "voz_ptbr.wav"
PREPARED_ARCHIVE_FILENAME = "dataset_preparado.tar.gz"
LANGUAGE = "Portuguese"
WHISPER_LANGUAGE = "pt"
WHISPER_MODEL = "medium"
NUM_EPOCHS = 5
LEARNING_RATE = 2e-6
REFERENCE_INDEX = 0
RANDOM_SEED = 42
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_SCALE = 0.30

gpu_upper = GPU_NAME.upper()
if "T4" in gpu_upper or GPU_MEMORY_GB < 20:
    MODEL_SIZE, MIXED_PRECISION, BATCH_SIZE = "0.6B", "no", 1
    TRAINING_SUPPORTED = True
elif "L4" in gpu_upper or GPU_MEMORY_GB < 40:
    MODEL_SIZE, MIXED_PRECISION, BATCH_SIZE = "1.7B", "bf16", 1
    TRAINING_SUPPORTED = True
else:
    MODEL_SIZE, MIXED_PRECISION, BATCH_SIZE = "1.7B", "bf16", 4
    TRAINING_SUPPORTED = True

MODEL_ID = f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-Base"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + f"_{SPEAKER_NAME}"
RUN_DIR = Path("/content/FineTunning-storage") / RUN_ID
for name in ["data/raw", "data/chunks", "samples", "checkpoints"]:
    (RUN_DIR / name).mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

raw_path = RUN_DIR / "data" / "raw" / RAW_AUDIO_FILENAME
if INPUT_MODE == "raw":
    sidebar_path = Path("/content") / RAW_AUDIO_FILENAME
    if sidebar_path.exists():
        shutil.copy2(sidebar_path, raw_path)
    else:
        print(f"Escolha {RAW_AUDIO_FILENAME} no seu Mac.")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError("Envie exatamente um arquivo WAV.")
        uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
        if not uploaded_name.lower().endswith(".wav"):
            raise ValueError("O arquivo precisa ter extensão .wav.")
        raw_path.write_bytes(uploaded_bytes)
elif INPUT_MODE == "prepared":
    print(f"Escolha o pacote {PREPARED_ARCHIVE_FILENAME} salvo na sessão anterior.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Envie exatamente um pacote .tar.gz.")
    archive_name, archive_bytes = next(iter(uploaded.items()))
    if not archive_name.endswith(".tar.gz"):
        raise ValueError("O pacote precisa terminar em .tar.gz.")
    archive_path = Path("/content") / archive_name
    archive_path.write_bytes(archive_bytes)
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(RUN_DIR, filter="data")
else:
    raise ValueError('INPUT_MODE deve ser "raw" ou "prepared".')

train_raw = RUN_DIR / "data" / "train_raw.jsonl"
train_codes = RUN_DIR / "data" / "train_with_codes.jsonl"
reference_path = RUN_DIR / "data" / "reference.wav"
reference_text_path = RUN_DIR / "data" / "reference.txt"
if INPUT_MODE == "prepared":
    required = [train_raw, train_codes, reference_path, reference_text_path]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Pacote preparado incompleto: {missing}")
    for manifest in [train_raw, train_codes]:
        rewritten = []
        for line in manifest.read_text(encoding="utf-8").splitlines():
            row = json.loads(line)
            row["audio"] = str((RUN_DIR / "data" / "chunks" / Path(row["audio"]).name).resolve())
            row["ref_audio"] = str(reference_path.resolve())
            rewritten.append(json.dumps(row, ensure_ascii=False))
        manifest.write_text("\n".join(rewritten) + "\n", encoding="utf-8")
    reference_text = reference_text_path.read_text(encoding="utf-8").strip()
    print("Dataset processado restaurado:", train_codes)

config = {
    "run_id": RUN_ID, "speaker_name": SPEAKER_NAME, "model_id": MODEL_ID,
    "gpu": GPU_NAME, "mixed_precision": MIXED_PRECISION,
    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS, "qwen_commit": QWEN_COMMIT,
    "input_mode": INPUT_MODE, "training_supported": TRAINING_SUPPORTED,
    "adaptation": "lora", "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
}
(RUN_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(config, ensure_ascii=False, indent=2))


## 2. Parte A — Preparação dos dados (`02_data_prep.ipynb`)

### 2.1 Inspecionar, normalizar e segmentar — original §§2–3

Primeiro conferimos duração, canais e sample rate. Em seguida, convertemos para mono/24 kHz e dividimos a gravação nas pausas. O treino trabalha melhor com exemplos curtos; manteremos somente trechos entre 2 e 15 segundos.

Escute os primeiros vinte segundos. Se houver distorção, eco forte ou ruído constante, regravar costuma ser melhor do que tentar “consertar” o áudio com filtros agressivos.


In [ ]:
if INPUT_MODE == "prepared":
    print("Etapa ignorada: o pacote já contém os dados processados.")
else:
    import numpy as np
    import soundfile as sf
    from IPython.display import Audio, display
    from pydub import AudioSegment
    from pydub.silence import split_on_silence

    info = sf.info(str(raw_path))
    print(info)
    if not 4 <= info.duration / 60 <= 12:
        print(f"ATENÇÃO: duração de {info.duration / 60:.1f} min; esperávamos 5–10 min.")

    normalized_path = RUN_DIR / "data" / "voz_24k_mono.wav"
    recording = AudioSegment.from_file(str(raw_path)).set_frame_rate(24_000).set_channels(1)
    recording.export(normalized_path, format="wav")
    waveform, sample_rate = sf.read(normalized_path)
    clipping = float(np.mean(np.abs(waveform) >= 0.99))
    print(f"Normalizado: {sample_rate} Hz | clipping: {clipping:.4%}")
    display(Audio(waveform[: sample_rate * 20], rate=sample_rate))

    chunks_dir = RUN_DIR / "data" / "chunks"
    raw_chunks = split_on_silence(
        recording,
        min_silence_len=550,
        silence_thresh=recording.dBFS - 18,
        keep_silence=120,
    )
    utterances = []
    for index, chunk in enumerate(raw_chunks):
        duration = len(chunk) / 1000
        if 2 <= duration <= 15:
            path = chunks_dir / f"utt_{index:04d}.wav"
            chunk.export(path, format="wav")
            utterances.append({"audio": path, "duration": duration})

    print(f"Trechos: {len(utterances)} | duração útil: {sum(x['duration'] for x in utterances) / 60:.1f} min")
    if len(utterances) < 20:
        raise RuntimeError("Poucos trechos. Verifique as pausas e a qualidade da gravação.")


### 2.2 Transcrever e revisar — original §§4–5

Whisper produz o texto associado a cada trecho. Esse texto é o rótulo supervisionado: uma transcrição errada ensina uma relação errada entre escrita e som. Depois da transcrição, ouviremos dez exemplos aleatórios.

Para corrigir um item, execute `utterances[ÍNDICE]["text"] = "texto correto"` antes de continuar.


In [ ]:
if INPUT_MODE == "prepared":
    print("Etapa ignorada: o pacote já contém os dados processados.")
else:
    import whisper

    whisper_model = whisper.load_model(WHISPER_MODEL, device="cuda")
    transcribed = []
    for index, item in enumerate(utterances, start=1):
        result = whisper_model.transcribe(
            str(item["audio"]), language=WHISPER_LANGUAGE,
            beam_size=5, temperature=0.0, condition_on_previous_text=False,
        )
        text = result["text"].strip()
        if len(text) >= 5:
            transcribed.append({**item, "text": text})
        if index % 10 == 0 or index == len(utterances):
            print(f"Transcritos: {index}/{len(utterances)}")

    utterances = transcribed
    del whisper_model
    torch.cuda.empty_cache()
    print("Trechos mantidos:", len(utterances))


In [ ]:
if INPUT_MODE == "prepared":
    print("Etapa ignorada: o pacote já contém os dados processados.")
else:
    for index in random.sample(range(len(utterances)), min(10, len(utterances))):
        item = utterances[index]
        print(f"Índice {index} | {item['duration']:.1f}s | {item['text']}")
        display(Audio(filename=str(item["audio"])))

    print("Correção: utterances[ÍNDICE]['text'] = 'Transcrição correta.'")


### 2.3 Criar JSONL e códigos acústicos — original §§6–7

Cada linha de `train_raw.jsonl` contém `audio`, `text` e `ref_audio`. Escolhemos um trecho limpo como referência e usamos o mesmo arquivo em todo o dataset, como recomenda o Qwen. Depois, o tokenizer de 12 Hz converte cada WAV em `audio_codes`.

Ouça a referência abaixo. Se ela contiver hesitação ou ruído, mude `REFERENCE_INDEX` na configuração e reexecute esta célula.


In [ ]:
if INPUT_MODE == "prepared":
    print("Etapa ignorada: o pacote já contém os dados processados.")
else:
    if not 0 <= REFERENCE_INDEX < len(utterances):
        raise IndexError("REFERENCE_INDEX não existe.")

    reference = utterances[REFERENCE_INDEX]
    reference_path = RUN_DIR / "data" / "reference.wav"
    shutil.copy2(reference["audio"], reference_path)
    reference_text = reference["text"]
    (RUN_DIR / "data" / "reference.txt").write_text(reference_text, encoding="utf-8")
    print("Referência:", reference_text)
    display(Audio(filename=str(reference_path)))

    train_raw = RUN_DIR / "data" / "train_raw.jsonl"
    with train_raw.open("w", encoding="utf-8") as handle:
        for item in utterances:
            row = {
                "audio": str(item["audio"].resolve()),
                "text": item["text"],
                "ref_audio": str(reference_path.resolve()),
            }
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("Linhas no manifesto:", len(utterances))


In [ ]:
if INPUT_MODE == "prepared":
    print("Etapa ignorada: o pacote já contém os dados processados.")
else:
    train_codes = RUN_DIR / "data" / "train_with_codes.jsonl"
    result = subprocess.run(
        [
            sys.executable, str(QWEN_REPO / "finetuning" / "prepare_data.py"),
            "--device", "cuda:0",
            "--tokenizer_model_path", "Qwen/Qwen3-TTS-Tokenizer-12Hz",
            "--input_jsonl", str(train_raw),
            "--output_jsonl", str(train_codes),
        ],
        text=True,
    )
    if result.returncode != 0 or not train_codes.exists():
        raise RuntimeError("Falha ao extrair audio_codes; consulte a saída acima.")
    print("Dataset pronto:", train_codes)


### 2.4 Salvar uma cópia do dataset processado

Se `INPUT_MODE = "raw"`, baixe este pacote antes de encerrar a sessão. Ele permite repetir LoRA na T4 ou comparar depois com full fine-tuning em L4/A100 sem executar Whisper novamente. No modo `prepared`, a célula apenas informa que o pacote já foi importado.


In [ ]:
if INPUT_MODE == "raw":
    prepared_export = Path("/content") / f"{RUN_ID}_dataset_preparado.tar.gz"
    subprocess.run(
        ["tar", "-czf", str(prepared_export), "-C", str(RUN_DIR), "data"],
        check=True,
    )
    print(f"Dataset preparado: {prepared_export} | {prepared_export.stat().st_size / 1024**2:.1f} MB")
    files.download(str(prepared_export))
else:
    print("Dataset já foi importado de um pacote preparado.")


## 3. Parte B — Clonagem zero-shot (`01_voice_cloning.ipynb`)

### 3.1 Carregar modelo, criar voice prompt e testar — original §§2–6

Esta é nossa linha de base antes do treino. O modelo recebe o WAV de referência e sua transcrição, cria um `voice_clone_prompt` e gera três frases inéditas. O original também demonstra presets, x-vector e histórias longas; eles foram omitidos porque não são necessários para avaliar o fine-tuning.

O primeiro carregamento baixa vários gigabytes. A T4 usa 0.6B; GPUs maiores usam 1.7B.


In [ ]:
from huggingface_hub import snapshot_download
from qwen_tts import Qwen3TTSModel
import soundfile as sf
from IPython.display import Audio, display

TEST_SENTENCES = [
    "Este é um teste de voz com uma frase que não apareceu durante o treinamento.",
    "Você confirmou a reunião para quinta-feira, às nove horas e trinta minutos?",
    "Que resultado incrível! Finalmente conseguimos concluir o experimento com sucesso.",
]
MODEL_PATH = Path(snapshot_download(MODEL_ID, cache_dir="/content/huggingface-cache"))
TORCH_DTYPE = {
    "no": torch.float32,
    "fp16": torch.float16,
    "bf16": torch.bfloat16,
}[MIXED_PRECISION]
base_model = Qwen3TTSModel.from_pretrained(
    str(MODEL_PATH), device_map="cuda:0",
    dtype=TORCH_DTYPE, attn_implementation="sdpa",
)
voice_prompt = base_model.create_voice_clone_prompt(
    ref_audio=str(reference_path), ref_text=reference_text
)


In [ ]:
for index, sentence in enumerate(TEST_SENTENCES, start=1):
    wavs, sr = base_model.generate_voice_clone(
        text=sentence, language=LANGUAGE, voice_clone_prompt=voice_prompt
    )
    output = RUN_DIR / "samples" / f"zero_shot_{index}.wav"
    sf.write(output, wavs[0], sr)
    print(sentence)
    display(Audio(wavs[0], rate=sr))

del base_model, voice_prompt
gc.collect()
torch.cuda.empty_cache()


## 4. Parte C — LoRA e teste (`03_finetune.ipynb`, variante leve)

### 4.1 Preparar e treinar os adaptadores

PEFT injeta matrizes LoRA nas projeções de atenção e MLP (`q/k/v/o`, `gate/up/down`). O Qwen-base permanece congelado. Usamos rank 8, batch 1, FP32 e acumulação de quatro passos na T4. Também aplicamos as correções comunitárias de projeção de texto e alinhamento de labels.

O checkpoint contém o adaptador e o embedding do locutor, não uma cópia completa do modelo. Na inferência, ele deve ser combinado com exatamente o mesmo modelo-base.


In [ ]:
finetuning_dir = RUN_DIR / "qwen_finetuning_lora"
shutil.copytree(QWEN_REPO / "finetuning", finetuning_dir, dirs_exist_ok=True)
train_script = finetuning_dir / "sft_12hz_lora.py"
source = train_script.read_text(encoding="utf-8")

old_accelerator = '''accelerator = Accelerator(
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        mixed_precision=None if args.mixed_precision == "no" else args.mixed_precision,
        log_with="tensorboard",
    )'''
new_accelerator = '''logging_dir = os.environ.get("QWEN_LOGGING_DIR", "./logs")
    os.makedirs(logging_dir, exist_ok=True)
    accelerator = Accelerator(
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        mixed_precision=None if args.mixed_precision == "no" else args.mixed_precision,
        log_with="tensorboard",
        project_dir=logging_dir,
    )'''
if old_accelerator not in source:
    raise RuntimeError("A implementação LoRA mudou na configuração do Accelerate.")
source = source.replace(old_accelerator, new_accelerator, 1)

old_load = '''qwen3tts = Qwen3TTSModel.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        attn_implementation=args.attn_implementation,
    )'''
new_load = '''model_dtype = {"no": torch.float32, "fp16": torch.float16, "bf16": torch.bfloat16}[args.mixed_precision]
    qwen3tts = Qwen3TTSModel.from_pretrained(
        MODEL_PATH,
        dtype=model_dtype,
        attn_implementation=args.attn_implementation,
    )'''
if old_load not in source:
    raise RuntimeError("A implementação LoRA mudou no carregamento do modelo.")
source = source.replace(old_load, new_load, 1)
source = source.replace(
    "optimizer = AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)",
    "optimizer = AdamW(model.parameters(), lr=args.lr, weight_decay=0.01, foreach=False)",
    1,
)
train_script.write_text(source, encoding="utf-8")
print("Treinador LoRA preparado:", train_script)


In [ ]:
train_log = RUN_DIR / "train_lora.log"
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["QWEN_LOGGING_DIR"] = str(RUN_DIR / "logs_lora")
command = [
    sys.executable, str(train_script),
    "--init_model_path", str(MODEL_PATH),
    "--output_model_path", str(RUN_DIR / "adapters"),
    "--train_jsonl", str(train_codes),
    "--batch_size", "1",
    "--gradient_accumulation_steps", "4",
    "--mixed_precision", MIXED_PRECISION,
    "--attn_implementation", "sdpa",
    "--lr", str(LEARNING_RATE),
    "--num_epochs", str(NUM_EPOCHS),
    "--speaker_name", SPEAKER_NAME,
    "--lora_rank", str(LORA_RANK),
    "--lora_alpha", str(LORA_ALPHA),
    "--lora_dropout", str(LORA_DROPOUT),
]

print("Iniciando LoRA. O modelo-base ficará congelado.")
with train_log.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(command, cwd=finetuning_dir, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log_handle.write(line)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError("Treinamento LoRA falhou. Consulte train_lora.log.")

checkpoints = sorted((RUN_DIR / "adapters").glob("checkpoint-epoch-*"),
                     key=lambda path: int(path.name.rsplit("-", 1)[-1]))
if not checkpoints:
    raise RuntimeError("Nenhum adaptador LoRA foi criado.")
print("Adaptadores:", [path.name for path in checkpoints])


### 4.2 Carregar o adaptador e comparar

O adaptador será aplicado ao mesmo Qwen3-TTS 0.6B. Começamos com escala LoRA 0,30, pois escala 1,0 pode exagerar a adaptação. Geramos as mesmas frases do zero-shot para comparar identidade, inteligibilidade, naturalidade e estabilidade. Depois, vale repetir com 0,20, 0,35 e 0,50.


In [ ]:
selected_checkpoint = checkpoints[-1]
infer_script = finetuning_dir / "infer_lora_custom_voice.py"
dtype_name = "fp32" if MIXED_PRECISION == "no" else MIXED_PRECISION

for index, sentence in enumerate(TEST_SENTENCES, start=1):
    output = RUN_DIR / "samples" / f"lora_{index}.wav"
    command = [
        sys.executable, str(infer_script),
        "--base_model_path", str(MODEL_PATH),
        "--adapter_path", str(selected_checkpoint),
        "--speaker_name", SPEAKER_NAME,
        "--text", sentence,
        "--language", LANGUAGE,
        "--output_wav", str(output),
        "--dtype", dtype_name,
        "--attn_implementation", "sdpa",
        "--lora_scale", str(LORA_SCALE),
        "--seed", str(RANDOM_SEED),
    ]
    subprocess.run(command, cwd=finetuning_dir, check=True)
    print(f"Frase {index} — zero-shot")
    display(Audio(filename=str(RUN_DIR / "samples" / f"zero_shot_{index}.wav")))
    print(f"Frase {index} — LoRA (escala {LORA_SCALE:.2f})")
    display(Audio(filename=str(output)))


## 5. Exportar os resultados para o Mac

O disco `/content` é temporário. Esta célula empacota configuração, áudio processado, logs, amostras e os dois checkpoints recentes em um `.tar` sem tentar recomprimir os pesos. Aguarde o download terminar antes de fechar a sessão.

Se o navegador bloquear um arquivo grande, encontre `/content/<run_id>.tar` no painel **Arquivos**, abra seu menu e escolha **Fazer download**.


In [ ]:
export_path = Path("/content") / f"{RUN_ID}.tar"
subprocess.run(
    ["tar", "-cf", str(export_path), "-C", str(RUN_DIR.parent), RUN_DIR.name],
    check=True,
)
print(f"Pacote: {export_path} | {export_path.stat().st_size / 1024**3:.2f} GB")
files.download(str(export_path))


## Apêndice — retomada de uma execução interrompida

A retomada não faz parte do primeiro fluxo porque adiciona complexidade e o otimizador oficial não é salvo. Um checkpoint concluído pode ser reutilizado, mas é necessário restaurar o `speaker_encoder` do modelo-base e reiniciar o otimizador. Isso não equivale exatamente a continuar do mesmo passo.

Para um primeiro corpus de 5–10 minutos, prefira concluir poucas épocas em uma única sessão e baixar o pacote final. Quando o fluxo básico estiver validado, a retomada pode ser adicionada como um notebook avançado separado.
